# 2 — A rede de regiões

## O que mudou, e por quê

Este projeto começou com **a pessoa como nó**: cada usuário era um vértice e cada aresta ligava
duas pessoas que se telefonaram. O problema é que **não temos a localização das pessoas** — temos
a localização da **antena** à qual a residência delas está associada. Todo mundo sob a mesma antena
tem exatamente as mesmas coordenadas e o mesmo quintil de renda.

Ou seja: a informação geográfica do projeto vive na antena, não na pessoa. Por isso a unidade de
análise passou a ser a **antena**, que aqui lemos como uma **região da cidade** (algo como um bairro).

| | Antes | Agora |
|---|---|---|
| **Nó** | uma pessoa | uma **antena** = uma região da cidade |
| **Aresta** | duas pessoas se telefonaram | **fluxo de chamadas** entre duas regiões |
| **Atributos do nó** | — | moradores, volume, quintil, insularidade, balanço |
| **Chamada dentro da mesma antena** | uma aresta comum | deixa de ser aresta → vira **insularidade** do nó |

Essa última linha é a mais importante e a menos óbvia. Se duas pessoas moram sob a mesma antena,
a ligação entre elas não atravessa nenhuma fronteira: ela não é um fluxo *entre* regiões. Em vez de
jogá-la fora, guardamos esse volume como um atributo da região — o quanto ela fala consigo mesma.
Em Campinas isso é **mais de um terço de todas as chamadas**, então seria um desperdício enorme
simplesmente descartar.

**O que este notebook faz:** constrói a rede de regiões passo a passo, mostra por que boa parte do
ferramental clássico de redes complexas deixa de fazer sentido nessa escala, e apresenta o que
entra no lugar.

## 1. Preparação

In [ ]:
import sys
from pathlib import Path

# permite rodar a partir de notebooks/ usando os módulos de src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
pd.set_option("display.max_columns", 40)

from src.utils import load_config
from src import antenna

CIDADE = "campinas"          # troque aqui: precisa existir config/<cidade>.yaml
config = load_config(CIDADE)

edges_antenna = pd.read_parquet(ROOT / config["data"]["edges_antenna_path"])
antennas = pd.read_parquet(ROOT / config["data"]["antennas_path"])

print(f"{len(edges_antenna):,} arestas usuário→usuário")
print(f"{len(antennas):,} antenas em {config['city_name']}")

## 2. De onde partimos

Dois arquivos, ambos gerados pelo notebook `1-eda`:

**`edges_antenna`** — uma linha por par de usuários que se falaram, já com a antena de cada ponta.
Ainda é o nível da pessoa; é a matéria-prima.

In [ ]:
edges_antenna.head()

| Coluna | O que é |
|---|---|
| `id_emisor` / `id_receiver` | os dois usuários (anonimizados) |
| `q_calls` | quantas chamadas o emissor fez para o receptor |
| `calls_duration_total` | duração total dessas chamadas |
| `residence_distance_km` | distância entre as residências dos dois |
| `emissor_antenna_id` / `receptor_antenna_id` | a antena de cada ponta |

**`antennas`** — uma linha por antena, com a geometria da residência e o quintil socioeconômico.

In [ ]:
antennas.head()

> **Um detalhe que muda a interpretação de tudo mais adiante:** o quintil está colado à
> `residence_geometry`, ou seja, **é um atributo da antena, não da pessoa**. Todos os moradores de
> uma mesma antena compartilham o mesmo quintil. Guarde isso — no notebook 4 essa observação vira
> o achado principal do projeto.

## 3. Passo 1 — quem mora onde

Antes de agregar qualquer coisa, precisamos saber a antena de residência de cada usuário. Um mesmo
usuário aparece em muitas linhas (como emissor e como receptor), então tomamos a **moda** das
antenas observadas para ele.

In [ ]:
user_antenna = antenna.build_user_antenna_map(edges_antenna)

print(f"usuários com antena conhecida: {len(user_antenna):,}")
print(f"antenas distintas usadas:      {user_antenna.nunique():,}")

moradores = user_antenna.value_counts()
print("\nmoradores por antena:")
print(moradores.describe().round(1))

Em Campinas: **25.176 moradores distribuídos em 145 antenas**, com mediana de **148 moradores por
região** (mínimo 11, máximo 587).

Esse é o preço da mudança de unidade: passamos de 25 mil nós para 145. Ganhamos geografia e
interpretabilidade, perdemos resolução individual. E a variação de tamanho é grande (11 a 587) —
por isso, mais adiante, várias métricas vão precisar ser normalizadas pelo tamanho da região.

## 4. Passo 2 — a tabela de nós

Agora agregamos as pessoas em suas regiões. Cada antena vira uma linha com os moradores somados.

In [ ]:
nodes = antenna.build_antenna_nodes(edges_antenna, antennas)
nodes.head()

As colunas que importam:

| Coluna | Como é calculada | O que significa |
|---|---|---|
| `n_users` | contagem de moradores | tamanho da região |
| `calls_out` / `calls_in` | soma de `q_calls` como emissor / receptor | volume emitido e recebido |
| `calls_internal` | soma de `q_calls` onde as duas pontas são a **mesma** antena | as chamadas que não saem da região |
| `calls_total` | `calls_out + calls_in − calls_internal` | volume que toca a região, contando as internas **uma vez só** |
| `insularity` | `calls_internal / calls_total` | fração do volume que fica dentro |
| `net_balance` | `(out − in) / (out + in)` | +1 = só emite, −1 = só recebe, 0 = equilíbrio |
| `calls_per_user` | `calls_total / n_users` | intensidade de uso, controlando o tamanho |

A subtração em `calls_total` é necessária porque uma chamada interna é contada duas vezes — uma em
`calls_out` e outra em `calls_in`, já que a antena é as duas pontas.

In [ ]:
nodes[["n_users", "calls_total", "calls_internal", "insularity",
       "net_balance", "calls_per_user"]].describe().round(3)

### O número que justifica a mudança de unidade

Quanto do volume da cidade some se simplesmente descartarmos as chamadas internas?

In [ ]:
internas = nodes["calls_internal"].sum()
mesma_antena = (edges_antenna["emissor_antenna_id"] == edges_antenna["receptor_antenna_id"])
total_cidade = edges_antenna["q_calls"].sum()

print(f"chamadas totais na base:        {total_cidade:>10,.0f}")
print(f"chamadas dentro da mesma antena:{internas:>10,.0f}  ({internas / total_cidade:.1%})")
print(f"chamadas entre regiões:         {total_cidade - internas:>10,.0f}  "
      f"({1 - internas / total_cidade:.1%})")
print(f"\narestas (pares de pessoas) intra-antena: {mesma_antena.mean():.1%}")

**35,9% de todas as chamadas da cidade não saem da região de origem.** Se tivéssemos apenas
descartado os laços internos ao agregar, jogaríamos fora mais de um terço dos dados — e, pior,
perderíamos justamente o indicador mais direto de "a vida acontece no bairro".

Por isso esse volume vira o atributo `insularity` de cada nó, e não lixo.

In [ ]:
mais_insulares = nodes.nlargest(8, "insularity")[
    ["antenna_id", "n_users", "calls_total", "insularity", "residence_quintile_state"]
].copy()
mais_insulares["insularity"] = (100 * mais_insulares["insularity"]).round(1).astype(str) + "%"
mais_insulares

A campeã de insularidade em Campinas fecha **55%** do seu volume dentro de si mesma. Repare que a
lista mistura quintis bem diferentes — insularidade não é privilégio de bairro rico nem de bairro
pobre; ela tem mais a ver com o quanto a região é autossuficiente em serviços e trabalho.

## 5. Passo 3 — a tabela de fluxos

Agora as arestas. Cada par de usuários é mapeado para o par de antenas correspondente, e os pares
com as duas pontas na mesma antena são retirados (já foram para `calls_internal`).

O caminho é: `A→B` e `B→A` viram um par não-direcionado de **pessoas**; esse par é mapeado para um
par de **regiões**; e todos os pares de pessoas entre as mesmas duas regiões são somados.

In [ ]:
flows = antenna.build_antenna_flows(edges_antenna, nodes)
flows.head()

| Coluna | O que é | Por que existe |
|---|---|---|
| `a`, `b` | as duas regiões (sempre `a < b`) | aresta não-direcionada |
| `q_calls` | total de chamadas trocadas, **nos dois sentidos** | é o **peso** da aresta |
| `n_pairs` | quantos pares de pessoas distintos estão por trás do fluxo | separa corredor **largo** (muita gente falando pouco) de **estreito** (poucos laços intensos) |
| `dist_km` | distância haversine entre as duas antenas | entra no modelo de gravidade (notebook 3) |
| `intensity` | `q_calls / (n_users_a × n_users_b)` | volume **per capita**: sem isso, o mapa de fluxos só redesenha onde mora mais gente |
| `weight` | cópia de `q_calls` | o peso usado por todos os algoritmos de rede |

> **Sobre a escolha do peso.** No nível de usuário, o peso era
> `log1p(q_calls) × log1p(duração)` — fazia sentido para medir a intensidade de um laço social
> entre duas pessoas. Entre regiões isso não se sustenta: somamos milhares de laços, e comprimir
> esse total em log destruiria justamente a informação de volume, que é o que interessa agora.
> O peso passou a ser **`q_calls` bruto**. A duração continua guardada na aresta, mas não entra
> no peso.

### Uma validação útil

`dist_km` é calculada por nós, a partir das coordenadas das antenas. Mas os dados originais já
traziam `residence_distance_km`, a distância entre as residências de cada par de pessoas. Se a
antena é mesmo um ponto fixo, as duas têm que bater exatamente.

In [ ]:
correlacao = flows[["dist_km", "mean_user_distance_km"]].corr().iloc[0, 1]
erro_max = (flows["dist_km"] - flows["mean_user_distance_km"]).abs().max()

print(f"correlação entre as duas distâncias: {correlacao:.6f}")
print(f"maior discrepância:                  {erro_max:.6f} km")

Correlação **1,000000** e discrepância praticamente nula. Isso confirma que todos os moradores de
uma antena têm literalmente as mesmas coordenadas — a antena é um ponto, não uma nuvem. É uma boa
notícia para a modelagem (a agregação não introduz erro espacial), mas também um lembrete de que
a resolução geográfica dos dados é a da antena, e não mais fina que isso.

## 6. O grafo

Com nós e arestas prontos, montamos o grafo. Também montamos a versão **dirigida**, que preserva
quem ligou para quem — necessária para reciprocidade e balanço.

In [ ]:
G = antenna.build_antenna_graph(flows, nodes)
D = antenna.build_antenna_digraph(antenna.build_directed_flows(edges_antenna, nodes), nodes)

graus = np.array([d for _, d in G.degree()])

print(f"nós (regiões):        {G.number_of_nodes()}")
print(f"arestas (fluxos):     {G.number_of_edges():,}")
print(f"densidade:            {nx.density(G):.3f}")
print(f"componentes conexas:  {nx.number_connected_components(G)}")
print(f"grau: mín {graus.min()}, mediana {np.median(graus):.0f}, máx {graus.max()}")
print(f"caminho mínimo médio: {nx.average_shortest_path_length(G):.2f}")

Guarde esses números, porque a próxima seção inteira decorre deles:

- **145 nós** — uma rede pequena.
- **densidade 0,557** — mais da metade de todos os pares possíveis de regiões tem contato.
- **uma única componente** — não há nada desconectado.
- **grau de 15 a 126, mediana 85** — todo mundo é vizinho de quase todo mundo.
- **caminho médio 1,44** — de qualquer região para qualquer outra em pouco mais de um passo.

## 7. Por que o ferramental antigo perde o sentido

Esta seção é importante para a apresentação: mostrar que sabemos **por que** removemos metade das
análises não é admitir fracasso, é demonstrar domínio do método. Vamos testar as três principais.

### 7.1 Lei de potência / rede livre de escala

A ideia de "rede livre de escala" depende de uma distribuição de grau de **cauda pesada**: muitos
nós com pouquíssimas conexões e alguns hubs com muitíssimas. Vamos olhar a nossa.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(graus, bins=25, color="steelblue")
axes[0].set_xlabel("grau (nº de regiões com quem tem contato)")
axes[0].set_ylabel("nº de regiões")
axes[0].set_title("Distribuição de grau")

x = np.sort(np.unique(graus))
ccdf = np.array([np.mean(graus >= k) for k in x])
axes[1].loglog(x, ccdf, "o", ms=5)
axes[1].set_xlabel("grau  k")
axes[1].set_ylabel("P(K ≥ k)")
axes[1].set_title("CCDF em log-log (uma reta indicaria cauda pesada)")
plt.tight_layout(); plt.show()

print(f"grau: média {graus.mean():.1f} ± {graus.std():.1f}  (amplitude {graus.min()}–{graus.max()})")
print(f"razão máx/mediana: {graus.max() / np.median(graus):.1f}x")

**Não há cauda.** O grau se concentra em torno de 80 com desvio de 26; o maior nó tem apenas 1,5×
a mediana. Numa rede livre de escala essa razão seria de ordens de grandeza. A CCDF, em vez de uma
reta descendente em log-log, despenca quase na vertical.

Ajustar uma lei de potência em 145 pontos sem cauda produziria um α sem significado. **Sai a lei
de potência**; entra a desigualdade de **volume** (seção 8) — que é onde a heterogeneidade real
desta rede está.

### 7.2 Small-world

O coeficiente σ compara a rede real com uma aleatória de mesmo tamanho: σ ≫ 1 significa
"agrupamento local muito alto **e** caminhos curtos". Na rede de usuários dava σ ≈ 596.

In [ ]:
C_real = nx.average_clustering(G)
L_real = nx.average_shortest_path_length(G)

R = nx.gnm_random_graph(G.number_of_nodes(), G.number_of_edges(), seed=42)
C_rand = nx.average_clustering(R)
L_rand = nx.average_shortest_path_length(R)

sigma = (C_real / C_rand) / (L_real / L_rand)
print(f"rede real:      clustering {C_real:.3f} | caminho médio {L_real:.3f}")
print(f"rede aleatória: clustering {C_rand:.3f} | caminho médio {L_rand:.3f}")
print(f"\nsigma = {sigma:.2f}   (na rede de usuários dava ~596)")

**σ ≈ 1,24** — ou seja, praticamente indistinguível de uma rede aleatória equivalente.

E o motivo é fácil de ver: com densidade 0,56, *qualquer* grafo com esse número de arestas terá
clustering alto (0,556 na aleatória!) e caminho médio curto (1,443 — idêntico ao real). Não há
nada de "mundo pequeno" a descobrir: a rede é pequena e densa, ponto. **Sai o small-world.**

### 7.3 k-core

O k-core busca o "núcleo" da rede podando repetidamente os nós de grau baixo.

In [ ]:
core = pd.Series(nx.core_number(G))
kmax = core.max()
print(f"k-core máximo: k = {kmax}")
print(f"regiões nesse core: {(core == kmax).sum()} de {G.number_of_nodes()} "
      f"({(core == kmax).mean():.0%} da rede)")

O núcleo máximo engloba **89 das 145 regiões — 61% da rede**. Um "núcleo" que contém a maioria dos
nós não separa nada; ele só reflete a densidade. **Sai o k-core**, entra o **s-core** (seção 10),
que poda por **força** (volume de chamadas) em vez de grau.

### Resumo

| Análise | Diagnóstico | Substituta |
|---|---|---|
| Lei de potência, CCDF de grau | sem cauda: grau 15–126 | desigualdade de **volume** (Lorenz/Gini) |
| Small-world (σ) | σ ≈ 1,2: igual ao acaso | — |
| k-core | 61% dos nós no core máximo | **s-core** (poda por força) |
| Componente gigante | uma componente só | — |
| Assortatividade de grau | grau não distingue nada | assortatividade por **quintil**, ponderada |

O padrão é sempre o mesmo: **na rede de regiões, a informação não está na topologia — está nos
pesos e no espaço.** Todo o resto do projeto segue essa pista.

## 8. O que entra no lugar (1): força e desigualdade de volume

Se o grau não distingue regiões, o que distingue? O **volume de chamadas**. Aqui a heterogeneidade
é enorme — e a curva de Lorenz é a forma padrão de medi-la.

In [ ]:
volume = nodes["calls_total"].to_numpy(dtype=float)

ordenado = np.sort(volume)
acumulado = np.concatenate([[0], np.cumsum(ordenado) / ordenado.sum()])
populacao = np.linspace(0, 1, len(ordenado) + 1)
gini = 2 * np.sum(np.arange(1, len(ordenado) + 1) * ordenado) / (len(ordenado) * ordenado.sum()) \
       - (len(ordenado) + 1) / len(ordenado)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(volume, bins=30, color="steelblue")
axes[0].set_xlabel("chamadas que tocam a região")
axes[0].set_ylabel("nº de regiões")
axes[0].set_title("Volume por região")
axes[1].plot(populacao, acumulado, color="crimson", lw=2, label="observado")
axes[1].plot([0, 1], [0, 1], "--", color="gray", lw=1, label="igualdade perfeita")
axes[1].set_xlabel("fração das regiões (da menor para a maior)")
axes[1].set_ylabel("fração acumulada do volume")
axes[1].set_title(f"Curva de Lorenz — Gini = {gini:.2f}")
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"volume: mínimo {volume.min():,.0f} | mediana {np.median(volume):,.0f} | "
      f"máximo {volume.max():,.0f}  →  razão máx/mín = {volume.max() / volume.min():.0f}x")

Aqui está a heterogeneidade que o grau não mostrava: a região de maior volume tem **212× o volume**
da menor, contra apenas 8× de diferença no grau. O **Gini de 0,37** é comparável à desigualdade de
renda de muitos países.

Leitura para a cidade: **todas as regiões falam uma com a outra, mas algumas falam muitíssimo mais.**
A estrutura da cidade está em quanto, não em com quem.

## 9. O que entra no lugar (2): o backbone

Numa rede com densidade 0,56, dizer que duas regiões "têm contato" não informa quase nada — quase
todas têm. A pergunta útil é outra: **quais fluxos são grandes demais para serem acaso?**

A resposta ingênua seria cortar pelos maiores pesos absolutos. Mas isso tem um viés claro: uma
região pequena nunca terá fluxos grandes, então ela seria apagada inteira do mapa, mesmo que
concentre quase toda a sua comunicação num único corredor.

O **filtro de disparidade** (Serrano, Boguñá & Vespignani, 2009) resolve isso julgando cada fluxo
contra a força do **próprio nó**. Para cada extremo `i` de uma aresta:

$$\alpha_{ij} = \left(1 - \frac{w_{ij}}{s_i}\right)^{k_i - 1}$$

onde $w_{ij}$ é o peso da aresta, $s_i$ a força do nó (soma dos pesos) e $k_i$ o grau. Isso é a
probabilidade de, repartindo a força do nó ao acaso entre seus vizinhos, obter um fluxo tão grande
quanto o observado. Se essa probabilidade for baixa (α < 0,05), o fluxo é significativo **para
aquele nó** — e a aresta entra no backbone.

A vantagem: o corredor principal de um bairro pequeno é preservado tanto quanto o de um grande.

In [ ]:
alpha = config.get("antenna", {}).get("backbone_alpha", 0.05)
B = antenna.disparity_filter(G, alpha=alpha)

peso_total = sum(d["weight"] for _, _, d in G.edges(data=True))
peso_backbone = sum(d["weight"] for _, _, d in B.edges(data=True))

print(f"fluxos originais: {G.number_of_edges():,}")
print(f"fluxos no backbone: {B.number_of_edges():,} "
      f"({B.number_of_edges() / G.number_of_edges():.1%} do total)")
print(f"volume preservado: {peso_backbone / peso_total:.1%} de todas as chamadas")
print(f"densidade: {nx.density(G):.3f} → {nx.density(B):.3f}")

**656 fluxos — 11% do total — carregam 62% de todas as chamadas.** A cidade tem um esqueleto de
comunicação bem definido.

Agora o teste que mostra por que o método importa: comparar com um corte ingênuo do mesmo tamanho.

In [ ]:
arestas_ordenadas = sorted(G.edges(data=True), key=lambda e: -e[2]["weight"])
corte_ingenuo = arestas_ordenadas[:B.number_of_edges()]

regioes_backbone = len([n for n, d in B.degree() if d > 0])
regioes_ingenuo = len({u for u, _, _ in corte_ingenuo} | {v for _, v, _ in corte_ingenuo})
peso_ingenuo = sum(d["weight"] for _, _, d in corte_ingenuo)

print(f"                     | regiões alcançadas | volume preservado")
print(f"filtro de disparidade|        {regioes_backbone:3d}         |      {peso_backbone / peso_total:.1%}")
print(f"corte pelos maiores  |        {regioes_ingenuo:3d}         |      {peso_ingenuo / peso_total:.1%}")

Com o **mesmo número de fluxos**, o corte ingênuo retém um pouco mais de volume (era de se esperar
— ele pega os maiores por construção), mas **deixa 4 regiões completamente fora do mapa**. O filtro
de disparidade cobre as 145.

É uma diferença modesta em Campinas, onde as regiões são relativamente homogêneas. Em cidades com
contraste maior entre centro e periferia, a diferença tende a ser bem mais dramática — e apagar
justamente as áreas pequenas seria um erro grave numa análise que quer informar política pública.

In [ ]:
corredores = pd.DataFrame(
    [(u, v, d["q_calls"], d["n_pairs"], round(d["dist_km"], 1)) for u, v, d in B.edges(data=True)],
    columns=["região A", "região B", "chamadas", "pares de pessoas", "distância (km)"],
).sort_values("chamadas", ascending=False)

print("Os 10 corredores mais fortes da cidade:")
corredores.head(10)

## 10. O que entra no lugar (3): macro-regiões funcionais

Detecção de comunidades sobre os **fluxos**: quais conjuntos de antenas conversam mais entre si do
que com o resto da cidade? É o que substitui as comunidades de usuários (que davam 426 grupos, sem
nenhuma leitura territorial).

Vale rodar com e sem peso, porque a comparação prova que o sinal está nos pesos.

In [ ]:
from networkx.algorithms.community import louvain_communities, modularity

com_pond = louvain_communities(G, weight="weight", seed=42)
com_topo = louvain_communities(G, weight=None, seed=42)

print(f"com peso: {len(com_pond)} macro-regiões | modularidade "
      f"{modularity(G, com_pond, weight='weight'):.3f} | tamanhos "
      f"{sorted((len(c) for c in com_pond), reverse=True)}")
print(f"sem peso: {len(com_topo)} grupos       | modularidade "
      f"{modularity(G, com_topo, weight=None):.3f} | tamanhos "
      f"{sorted((len(c) for c in com_topo), reverse=True)}")

**Com peso: 5 macro-regiões, modularidade 0,40. Sem peso: modularidade 0,09** — praticamente nada.

Isso confirma o diagnóstico da seção 7: a topologia sozinha não carrega informação nenhuma nesta
rede; toda a estrutura está nos pesos.

> **Atenção ao comparar modularidades.** Na rede de usuários dava Q = 0,98, e seria tentador dizer
> que 0,40 é "pior". Não é: numa rede esparsa, qualquer partição decente dá Q altíssimo. Numa rede
> com densidade 0,56, **0,40 é uma divisão nítida**. As duas escalas não são comparáveis.

In [ ]:
region_of = {a: i for i, com in enumerate(com_pond) for a in com}
nodes["macro_region"] = nodes["antenna_id"].map(region_of)

nodes.groupby("macro_region").agg(
    regioes=("antenna_id", "size"),
    moradores=("n_users", "sum"),
    chamadas=("calls_total", "sum"),
    insularidade_media=("insularity", "mean"),
).round(3)

O teste decisivo dessas macro-regiões está no **notebook 3**: elas saem **espacialmente contíguas**
no mapa, mesmo o Louvain não sabendo absolutamente nada sobre geografia. É o resultado visual mais
forte do projeto.

> ⚠️ **Uma ressalva honesta:** o peso é o volume bruto de chamadas, que cresce com o tamanho da
> região. Então parte do que o Louvain agrupa reflete densidade populacional, não só afinidade.
> A coluna `intensity` (volume per capita) está calculada nos fluxos justamente para permitir
> refazer esse teste — vale a pena comparar as duas partições.

## 11. O que entra no lugar (4): s-core

O **s-core** é o k-core generalizado para pesos (Eidsaa & Almaas, 2013): em vez de podar nós por
grau, poda por **força** — a soma dos pesos das suas arestas. Enquanto o k-core trivializava
(61% dos nós no core máximo), o s-core encontra o núcleo real de tráfego da cidade.

In [ ]:
niveis = antenna.s_core_levels(G)
grade = np.linspace(0, niveis.max(), 40)
tamanhos = [int((niveis >= t).sum()) for t in grade]

plt.figure(figsize=(7.5, 4.5))
plt.plot(grade, tamanhos, "o-", color="darkorange")
plt.xlabel("limiar de força (chamadas)")
plt.ylabel("regiões que sobrevivem")
plt.title("Decomposição s-core")
plt.show()

print(f"núcleo final: {(niveis >= niveis.max()).sum()} regiões "
      f"(força mínima {niveis.max():,.0f} chamadas)")

O núcleo final reúne **42 das 145 regiões** — bem mais seletivo que os 89 do k-core, e agora com
uma leitura clara: são as áreas que sustentam o grosso da comunicação da cidade. É essa lista que
alimenta a discussão de resiliência no notebook 4.

## 12. Direção dos fluxos: reciprocidade e balanço

Tudo até aqui usou o grafo não-direcionado. Mas a versão dirigida responde uma pergunta que a
outra não responde: **quais regiões emitem mais do que recebem?**

Primeiro, vale checar se a direção sequer importa.

In [ ]:
reciprocidade = nx.reciprocity(D)
print(f"reciprocidade: {reciprocidade:.1%}")
print("(fração dos fluxos A→B que têm um B→A correspondente)")

**86%** — quem recebe, devolve. A comunicação entre regiões é fortemente equilibrada, o que já é
um achado: não existem regiões que só recebem ligações sem retornar.

Mas o equilíbrio não é perfeito, e o desvio é informativo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(nodes["net_balance"].dropna(), bins=25, color="teal")
axes[0].axvline(0, color="black", ls="--", lw=1)
axes[0].set_xlabel("balanço líquido  (emitidas − recebidas) / total")
axes[0].set_ylabel("nº de regiões")
axes[0].set_title("Emissoras (>0) vs. receptoras (<0)")
axes[1].hist(nodes["insularity"].dropna(), bins=25, color="indianred")
axes[1].set_xlabel("insularidade")
axes[1].set_ylabel("nº de regiões")
axes[1].set_title("O quanto cada região fala consigo mesma")
plt.tight_layout(); plt.show()

print("regiões mais receptoras (recebem bem mais do que emitem):")
print(nodes.nsmallest(5, "net_balance")[
    ["antenna_id", "n_users", "net_balance", "calls_total"]].to_string(index=False))

Regiões consistentemente **receptoras** são candidatas naturais a concentrar emprego, comércio e
serviços — é para lá que as pessoas ligam. Regiões **emissoras** tendem a ser majoritariamente
residenciais. O mapa desse balanço está no notebook 3.

## Síntese

Construímos a rede de regiões e mostramos, com teste em vez de afirmação, por que o ferramental de
redes esparsas não se aplica aqui:

| | Campinas |
|---|---|
| Regiões / fluxos / densidade | 145 / 5.817 / 0,557 |
| Chamadas que não saem da região | **35,9%** |
| Desigualdade de volume (Gini) | 0,37 |
| Backbone (α = 0,05) | 656 fluxos (11%) com **62%** do volume |
| Macro-regiões funcionais | **5** (modularidade 0,40) |
| Núcleo s-core | 42 regiões |
| Reciprocidade | 86% |

**Próximo passo:** o notebook 3 leva tudo isso para o mapa e introduz o modelo de gravidade.